# Sentiment Analysis Benchmark: Jev vs Claude Haiku 5.5

This notebook compares two models on customer-message sentiment classification (**positive / negative / neutral**):

| Model | Access | Structured output method |
|---|---|---|
| **Jev** (`typesafe/jev-1.13`) | OpenRouter Decisions API | `choice` question; returns per-label probabilities |
| **Claude Haiku 5.5** | Amazon Bedrock Converse API | Forced tool call with an `enum` JSON schema |

For each message, the notebook records:
- **Predicted sentiment** from each model, compared against a reference label
- **Inference latency** (ms), measured on the client with `time.perf_counter()`
- **Token usage** (input and output tokens per call)

**Output:** `sentiment_comparison_with_tokens.csv`, with one row per message.

**dataset used:** The following dataset has been used to predict and also measure accuracy of sentiment analysis - https://www.kaggle.com/datasets/kundanbedmutha/customer-sentiment-dataset

**Prerequisites**
- The `OPENROUTER_API_KEY` environment variable is set
- AWS credentials with `bedrock:InvokeModel` permission, and model access enabled for Haiku 5.5
- An input CSV (`sample_customer_sentiment_data.csv`) with columns: `content`, `reference_sentiment`

## 1. Setup and configuration

Imports the libraries, reads the OpenRouter API key, and creates the Bedrock runtime client.

- `MODEL_ID`: the Haiku 5.5 cross-region inference profile (`us.` = US regional routing)
- The Bedrock client uses your standard AWS credentials (profile, SSO or IAM role), so no API key is needed

In [ ]:
#!pip install -q requests
#!pip install boto3, botocore
#!pip install tqdm ipywidgets
#!pip install pandas

In [ ]:
import requests
import boto3
import time
from datetime import datetime
import csv
from tqdm.auto import tqdm

In [ ]:
MODEL_ID = "us.anthropic.claude-haiku-5-5"
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OpenRouter API key: ")

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

## 2. Jev sentiment function: `check_sentiment_using_jev(message)`

Sends a **choice** question to the Jev Decisions API and picks the label with the **highest probability**.

**How it works**
1. Builds a `choice` question whose `criteria` describe each sentiment class. The descriptions steer the model.
2. Times the HTTP call using `time.perf_counter()` before and after.
3. Selects the top label with `max(probs, key=probs.get)`.
4. Reads token usage from the `usage` field of the response, if it's present.

**Returns:** `(label, latency_ms, input_tokens, output_tokens)`

| Value | Type | Example |
|---|---|---|
| `label` | `str` | `"negative"` |
| `latency_ms` | `float` | `114.6` |
| `input_tokens` | `int` or `None` | `98` |
| `output_tokens` | `int` or `None` | `0` |

> ⚠️ If the token columns come out empty, the API uses different usage field names. Run `print(r.json())` once to check.

In [ ]:
def check_sentiment_using_jev(message: str):
    start = time.perf_counter()
    r = requests.post(
        "https://openrouter.ai/api/alpha/decisions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "typesafe/jev-1.13",
            "state": {"message": message},
            "questions": {
                "sentiment": {
                    "type": "choice",
                    "instructions": "What is the overall sentiment of `message`?",
                    "criteria": {
                        "positive": "Satisfied, happy, praising, grateful, optimistic",
                        "negative": "Unhappy, angry, complaining, disappointed, worried",
                        "neutral": "Factual, a question or request, mixed, or no clear emotion",
                    },
                }
            },
        },
        timeout=10,
    )
    r.raise_for_status()
    data = r.json()
    
    probs = data["answers"]["sentiment"]["probabilities"]   # e.g. {"positive": 0.1, "negative": 0.8, "neutral": 0.1}
    best = max(probs, key=probs.get)
    
    end = time.perf_counter()

    usage = data.get("usage", {})   # field names may differ, see note below
    in_tok = usage.get("prompt_tokens") or usage.get("input_tokens")
    out_tok = usage.get("completion_tokens") or usage.get("output_tokens")
    
    return best, round((end - start) * 1000, 1), in_tok, out_tok

## 3. Structured output schema for Haiku: `SENTIMENT_TOOL`

Defines a tool (`record_sentiment`) whose JSON schema allows **only** `positive`, `negative` or `neutral`.

Forcing the model to call this tool guarantees **valid, machine-readable output**: no free text and no labels outside the list. The `description` on the field gives the same class definitions used for Jev, so the comparison is fair.

In [ ]:
SENTIMENT_TOOL = {
    "toolSpec": {
        "name": "record_sentiment",
        "description": "Record the overall sentiment of the message.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "negative", "neutral"],
                        "description": (
                            "positive: satisfied, happy, praising, grateful, optimistic; "
                            "negative: unhappy, angry, complaining, disappointed, worried; "
                            "neutral: factual, a question or request, mixed, or no clear emotion"
                        ),
                    }
                },
                "required": ["sentiment"],
            }
        },
    }
}

## 4. Haiku 5.5 sentiment function: `check_sentiment_using_haiku(message)`

Calls Claude Haiku 5.5 through the Bedrock **Converse API** and forces it to call `record_sentiment`.

**How it works**
1. `toolChoice = {"tool": {"name": "record_sentiment"}}` makes the model answer through the schema.
2. `maxTokens=50` is enough, because the output is a single label. 
3. Times the call with `time.perf_counter()`.
4. Extracts the label from the `toolUse` block.
5. Reads the exact token counts from `resp["usage"]`. The input count includes the system prompt and tool schema overhead.

**Returns:** `(label, latency_ms, input_tokens, output_tokens)`, the same shape as the Jev function.

In [ ]:
def check_sentiment_using_haiku(message: str):
    start = time.perf_counter()
    resp = bedrock.converse(
        modelId=MODEL_ID,
        system=[{"text": "You classify the sentiment of customer messages."}],
        messages=[{"role": "user", "content": [{"text": message}]}],
        toolConfig={"tools": [SENTIMENT_TOOL],
                    "toolChoice": {"tool": {"name": "record_sentiment"}}},
        inferenceConfig={"maxTokens": 50},
    )
    latency_ms = round((time.perf_counter() - start) * 1000, 1)

    label = next(b["toolUse"]["input"]["sentiment"]
                 for b in resp["output"]["message"]["content"] if "toolUse" in b)

    in_tok = resp["usage"]["inputTokens"]
    out_tok = resp["usage"]["outputTokens"]
    return label, latency_ms, in_tok, out_tok

## 5. Run the benchmark and write the results CSV

Reads the input CSV row by row, sends each message to **both** models, and writes one result row per message.

**Details**
- **Sequential calls:** the models are called one after the other, never in parallel, so latency isn't affected by queuing or throttling.
- **`fout.flush()`:** saves each row to disk right away, so finished rows survive a crash.
- **`MAX_ROWS`:** limits the run. Lower it for quick tests.
- **`utf-8-sig`:** makes accented characters display correctly in Excel.

**Output columns**

| Column | Description |
|---|---|
| `content` | Original message |
| `reference_sentiment` | Ground-truth label |
| `jev_predicted_sentiment` / `haiku_predicted_sentiment` | Model predictions |
| `jev_inference_latency_ms` / `haiku_inference_latency_ms` | Client-side latency |
| `jev_input_tokens` / `jev_output_tokens` | Jev token usage |
| `haiku_input_tokens` / `haiku_output_tokens` | Haiku token usage |

In [ ]:
input_file = "sample_customer_sentiment_data.csv"
output_file = "discriminative_jev_vs_generative_haiku55_sentiment.csv"
MAX_ROWS = 1000

# Count the rows first so the bar can show the % done and time remaining
with open(input_file, newline="", encoding="utf-8") as f:
    total_rows = min(sum(1 for _ in f) - 1, MAX_ROWS)   # minus the header

jev_correct = haiku_correct = 0
jev_ms_sum = haiku_ms_sum = 0.0

FIELDS = [
    "content", "reference_sentiment",
    "jev_predicted_sentiment", "haiku_predicted_sentiment",
    "jev_inference_latency_ms", "haiku_inference_latency_ms",
    "jev_input_tokens", "jev_output_tokens",
    "haiku_input_tokens", "haiku_output_tokens",
]

with open(input_file, newline="", encoding="utf-8") as fin, \
     open(output_file, "w", newline="", encoding="utf-8-sig") as fout:
    reader = csv.reader(fin)
    next(reader)
    writer = csv.writer(fout)
    writer.writerow(FIELDS)

    with tqdm(total=total_rows, desc="Benchmarking", unit="msg") as pbar:
        for i, row in enumerate(reader, start=1):
            content, ref = row[0], row[1]
            j_label, j_ms, j_in, j_out = check_sentiment_using_jev(content)
            h_label, h_ms, h_in, h_out = check_sentiment_using_haiku(content)

            writer.writerow([content, ref, j_label, h_label, j_ms, h_ms,
                             j_in, j_out, h_in, h_out])
            fout.flush()

            # Running stats, updated next to the bar
            jev_correct += (j_label == ref)
            haiku_correct += (h_label == ref)
            jev_ms_sum += j_ms
            haiku_ms_sum += h_ms
            pbar.set_postfix({
                "jev_acc": f"{jev_correct / i:.1%}",
                "haiku_acc": f"{haiku_correct / i:.1%}",
                "jev_ms": f"{jev_ms_sum / i:.0f}",
                "haiku_ms": f"{haiku_ms_sum / i:.0f}",
            })
            pbar.update(1)

            if i >= MAX_ROWS:
                break

print(f"Saved {i} rows to {output_file}")

## 6. Results summary

Loads the benchmark CSV and compares the two models on:
- **Accuracy:** % of predictions that match `reference_sentiment` (failed calls excluded)
- **Average latency:** mean client-side inference time in ms
- **Average tokens:** mean input and output tokens per call

The optional cell below breaks accuracy down by class and shows a confusion matrix, which tells you which labels each model confuses.

In [ ]:
import pandas as pd

results_file = "discriminative_jev_vs_generative_haiku55_sentiment.csv"
df = pd.read_csv(results_file, encoding="utf-8-sig")

MODELS = {"Jev": "jev", "Haiku 5.5": "haiku"}

def norm(s):
    """Lower-case and trim labels so 'Positive ' matches 'positive'."""
    return s.astype(str).str.strip().str.lower()

ref = norm(df["reference_sentiment"])
summary = {}

for name, p in MODELS.items():
    pred = norm(df[f"{p}_predicted_sentiment"])
    valid = ~pred.str.startswith("error")            # skip rows where the call failed

    summary[name] = {
        "rows_scored":        int(valid.sum()),
        "accuracy_%":         round((pred[valid] == ref[valid]).mean() * 100, 2),
        "avg_latency_ms":     round(pd.to_numeric(df[f"{p}_inference_latency_ms"], errors="coerce").mean(), 1),
        "avg_input_tokens":   round(pd.to_numeric(df[f"{p}_input_tokens"], errors="coerce").mean(), 1),
        "avg_output_tokens":  round(pd.to_numeric(df[f"{p}_output_tokens"], errors="coerce").mean(), 1),
    }

summary_df = pd.DataFrame(summary)
display(summary_df)          # in a script, use print(summary_df) instead